This page aims to experiment with warmup lengths on two different MVNs.

In [ ]:
!pip uninstall -yq jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt tensorflow-probability
# heads out since jax might drop support for cuda 12, currently (as of Aug 10, 2026), JAX has issues with CUDA 13
# see this post: https://github.com/jax-ml/jax/issues/37923
!pip install -Uq "jax[cuda12]" tfp-nightly blackjax inference_gym optax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.9/390.9 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 138.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow-probability>=0.13.0, which is not installed.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.2 which is incompatible.


In [ ]:
# run those checks if package compatibility is in trouble

# import jax
# import tensorflow_probability as tfp
# import jaxlib

# print("jaxlib:", jaxlib.__version__)
# print("TFP:", tfp.__version__)

# !pip show jax
# !pip show jaxlib
# !pip show blackjax

# import jax.numpy as jnp

# import blackjax

# import tensorflow_probability.substrates.jax as tfp
# import inference_gym.using_jax as gym

# print("JAX:", jax.__version__)
# print("BlackJAX:", blackjax.__version__)
# print("TFP:", tfp.__version__)
# print("ArviZ:", avs.__version__)
# print("Inference Gym imported successfully!")

**Package Import and other setups**

In [ ]:
# GPU set up to accelerate performance
import os
# in case jax eats up my GPU RAM
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from jax import random, jit, vmap, lax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
import inference_gym.using_jax as gym
import jaxlib
import blackjax

import optax

# import arviz as az
# import arviz_stats as avs

import warnings
warnings.filterwarnings('ignore')

import psutil

import gc

from google.colab import drive
from matplotlib.lines import Line2D

process = psutil.Process(os.getpid())

def mem(msg):
    print(f"{msg}: {process.memory_info().rss / 1024**2:.1f} MB")

# verification to make sure this is on a GPU
print(jax.devices())
print(jax.default_backend())

[CudaDevice(id=0)]
gpu


In [ ]:
drive.mount('/content/drive')
utility_link = '/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/AdditionalExperiment/MVN_Experiment.py'
with open(utility_link) as f: exec(f.read())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Hyperparam Setups**

In [ ]:
max_warmup = 1000
warmup_window = 100

window_array = np.append(np.repeat(10, 10),
                      np.repeat(warmup_window, max_warmup // warmup_window - 1))

warmup_length = np.repeat(10, len(window_array))
for i in range(len(warmup_length) - 1):
    warmup_length[i + 1] = warmup_length[i] + window_array[i + 1]

# Transition kernel for short regime
repitition = 10
num_chains_short = 2048
num_super_chains = 16

In [ ]:
# quantiles for chi squared with df = 1
chi_up = 3.841459 # 95th quantile for chi squared with df = 1
chi_lo = 0.00393214  # 05th quantile for chi squared with df = 1
tau = 1e-4
M = num_chains_short // num_super_chains
nRhat_lower = np.sqrt(1 + 1 / M + tau)
eps_lower = nRhat_lower - 1
bound = [chi_lo / num_chains_short, chi_up / num_chains_short]
threshold = eps_lower

**MVN with isotropic covariance**

In [ ]:
num_dim = 100
# mean array:
mu = jnp.full(num_dim, 1.3)
# covariance matrix:
cov = jnp.eye(num_dim)
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=cov)
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape + (num_dim,))

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
Iso_MSE_c_list = []
Iso_MSE_n_list = []
Iso_RHat_c_list = []
Iso_RHat_n_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_c_list,Iso_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, Iso_RHat_n_list,Iso_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1768.2 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.0005612447275780141
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0005232213297858834
Simulation Start: 2166.7 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0004754156689159572
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0004897068138234317
Simulation Start: 2174.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004823815543204546
Naive initialization. Warmup Length: 30; mean of MSE is: 0.0004912347067147493
Simulation Start: 2185.4 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004670104826800525
Naive initialization. Warmup Length: 40; mean of MSE is: 0.0004935162141919136
Simulation Start: 2203.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.00045773471356369555
Naive initialization. Warmup Length: 50; mean of MSE is: 0.0004582492110785097
Simulation Start: 2228.5 MB
Constrained

**Save to DataFrame and pickle**

In [ ]:
MSE_c_df = pd.DataFrame(Iso_MSE_c_list)
R_Hat_c_df = pd.DataFrame(Iso_RHat_c_list)

MSE_n_df = pd.DataFrame(Iso_MSE_n_list)
R_Hat_n_df = pd.DataFrame(Iso_RHat_n_list)

In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/Iso_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/Iso_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/Iso_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/Iso_Rhat_n.pkl"
)

**MVN with AR(1) covariance matrix, $\rho = 0.5$**

In [ ]:
num_dim = 100
rho = 0.5
# mean array:
mu = jnp.full(num_dim, 1.3)
# covariance matrix:
cov = [ [rho**(abs(i-j)) for j in range(num_dim)] for i in range(num_dim)]
target = tfd.MultivariateNormalFullCovariance(
      loc=jnp.array(mu),
      covariance_matrix=jnp.array(cov))
init_step_size = 0.5
def target_log_prob_fn(x):
    return target.log_prob(x)
def initialize(shape, key):
    return random.normal(key, shape + (num_dim,))
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)

mean_benchmark = target.mean()
var_benchmark = target.variance()

In [ ]:
#simulation part:
AR_MSE_c_list = []
AR_MSE_n_list = []
AR_RHat_c_list = []
AR_RHat_n_list = []

In [ ]:
base_key = random.PRNGKey(0)
keys = random.split(base_key, repitition)
for length in warmup_length:
  mem(f"Simulation Start")
  simulation(length,num_chains_short, num_super_chains,
             False,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, AR_RHat_c_list,AR_MSE_c_list,
             mean_benchmark,var_benchmark)
  simulation(length,num_chains_short, num_super_chains,
             True,initialize, keys,
             target_log_prob_fn,init_step_size,
             repitition, AR_RHat_n_list,AR_MSE_n_list,
             mean_benchmark,var_benchmark)

Simulation Start: 1698.4 MB
Constrained initialization. Warmup Length: 10; mean of MSE is: 0.00047791399993002415
Naive initialization. Warmup Length: 10; mean of MSE is: 0.0004974481416866183
Simulation Start: 2092.8 MB
Constrained initialization. Warmup Length: 20; mean of MSE is: 0.0005201572203077376
Naive initialization. Warmup Length: 20; mean of MSE is: 0.0005099233239889145
Simulation Start: 2102.2 MB
Constrained initialization. Warmup Length: 30; mean of MSE is: 0.0004982638056389987
Naive initialization. Warmup Length: 30; mean of MSE is: 0.000538877968210727
Simulation Start: 2116.2 MB
Constrained initialization. Warmup Length: 40; mean of MSE is: 0.0004871022538281977
Naive initialization. Warmup Length: 40; mean of MSE is: 0.00047924643149599433
Simulation Start: 2133.2 MB
Constrained initialization. Warmup Length: 50; mean of MSE is: 0.0004867121169809252
Naive initialization. Warmup Length: 50; mean of MSE is: 0.00047169424942694604
Simulation Start: 2157.0 MB
Constraine

In [ ]:
MSE_c_df = pd.DataFrame(AR_MSE_c_list)
R_Hat_c_df = pd.DataFrame(AR_RHat_c_list)

MSE_n_df = pd.DataFrame(AR_MSE_n_list)
R_Hat_n_df = pd.DataFrame(AR_RHat_n_list)


In [ ]:
MSE_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/AR_MSE_c.pkl"
)

MSE_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/AR_MSE_n.pkl"
)

R_Hat_c_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/AR_Rhat_c.pkl"
)

R_Hat_n_df.to_pickle(
    "/content/drive/MyDrive/Colab Notebooks/2026_Summer_MCMC/MVN_pkl_files/AR_Rhat_n.pkl"
)